In [4]:
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
from nemosis import static_table, dynamic_data_compiler, defaults
import plotly.express as px
import os
import glob
import dask.dataframe as dd
import re

# raw_data_cache = '/Volumes/T7/Misc'

pd.set_option('display.max_columns', None)

In [ ]:
# # Download volume bid data in year increments
# volume_bids = dynamic_data_compiler(start_time='2015/07/01 00:00:00',
#                                    end_time='2024/01/31 00:00:00',
#                                    table_name='BIDPEROFFER_D',
#                                    raw_data_location=raw_data_cache)

NameError: name 'raw_data_cache' is not defined

In [ ]:
# import os

# # Define the directory where the sorted files are now located
# sorted_dir = "/Volumes/T7/bid-volume-data-sorted"

# # Iterate through all files in the sorted directory
# for filename in os.listdir(sorted_dir):
#     if filename.startswith("._") or not filename.endswith(".feather"):
#         continue  # Skip hidden macOS files and non-feather files

#     file_path = os.path.join(sorted_dir, filename)

#     # Extract filename without extension
#     name_part, ext = os.path.splitext(filename)  # Splits into name and '.feather'

#     # Debugging info: print filenames being checked
#     print(f"Checking: {filename}")

#     # Check if filename ends with '0000' before '.feather'
#     if name_part.endswith("0000"):
#         new_name_part = name_part[:-4]  # Remove last four zeroes
#         new_filename = new_name_part + ext  # Append '.feather'

#         new_file_path = os.path.join(sorted_dir, new_filename)

#         # Check if the new filename already exists before renaming
#         if os.path.exists(new_file_path):
#             print(f"Skipping {filename}, {new_filename} already exists.")
#             continue

#         # Rename the file
#         os.rename(file_path, new_file_path)
#         print(f"Renamed: {filename} -> {new_filename}")

# print("Filename cleanup complete.")

In [5]:
# # Filter feather files for RaiseReg and LowerReg BIDTYPEs and save as parquet files
# # Create the output directory if it doesn't already exist
# output_dir = '/Volumes/T7/bid-volume-filtered-2'
# os.makedirs(output_dir, exist_ok=True)

# file_list = glob.glob('/Volumes/T7/bid-volume-data-sorted/*.feather')

# # Sort alphabetically by filename
# file_list.sort()

# for file in file_list:
#     print(f"Processing {file}...")
#     # Read the Feather file
#     df = pd.read_feather(file)
    
#     # Filter to keep rows where BIDTYPE is either "RAISEREG" or "LOWERREG"
#     filtered_volume_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

#     # Print the first few rows of the filtered DataFrame
#     print("Filtered DataFrame head:")
#     print(filtered_volume_bids.head())

#     # Get just the filename without the path
#     base_filename = os.path.basename(file)  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000.feather"
#     # Remove '.feather' extension
#     filename_no_ext = os.path.splitext(base_filename)[0]  # e.g. "PUBLIC_DVD_BIDPEROFFER_D_201201010000"

#     # Construct the full output path in output_dir
#     out_file = os.path.join(output_dir, filename_no_ext + ".parquet")

#     # Save to Parquet
#     filtered_volume_bids.to_parquet(out_file, index=False)
#     print(f"Finished processing {file} -> {out_file}\n")

In [ ]:
# # Filter for rows where INTERVAL_DATETIME contains the bids for the 6-6:05pm time interval

# def filter_interval_time(input_dir, output_dir, target_time='18:05:00'):
#     """
#     Filter parquet files for rows where INTERVAL_DATETIME contains a specific time (e.g., '18:05:00')
    
#     Parameters:
#     -----------
#     input_dir : str
#         Directory containing the parquet files to filter
#     output_dir : str
#         Directory where filtered files will be saved
#     target_time : str
#         Time to filter for in format 'HH:MM:SS'
#     """
#     print(f"Starting to filter data for interval time: {target_time}")
    
#     # Create the output directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
#     print(f"Output directory created/verified: {output_dir}")
    
#     # Get list of all parquet files in the input directory
#     file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    
#     # Sort the file list to ensure consistent processing order
#     file_list.sort()
    
#     print(f"Found {len(file_list)} parquet files to process")
    
#     # Counter for tracking progress
#     total_files = len(file_list)
#     files_processed = 0
#     rows_filtered = 0
#     total_rows_processed = 0
#     files_with_target = 0
    
#     for file_path in file_list:
#         file_name = os.path.basename(file_path)
#         files_processed += 1
        
#         print(f"Processing file {files_processed}/{total_files}: {file_name}")
        
#         try:
#             # Read the parquet file
#             df = pd.read_parquet(file_path)
            
#             # Track total rows for statistics
#             file_row_count = len(df)
#             total_rows_processed += file_row_count
            
#             # Filter for rows where INTERVAL_DATETIME contains the target time
#             if 'INTERVAL_DATETIME' in df.columns:
#                 # Check data type and filter accordingly
#                 if pd.api.types.is_datetime64_any_dtype(df['INTERVAL_DATETIME']):
#                     # If it's a datetime column, extract components
#                     hour, minute, second = map(int, target_time.split(':'))
#                     mask = (df['INTERVAL_DATETIME'].dt.hour == hour) & \
#                            (df['INTERVAL_DATETIME'].dt.minute == minute) & \
#                            (df['INTERVAL_DATETIME'].dt.second == second)
#                 else:
#                     # If it's a string column, use string contains method
#                     mask = df['INTERVAL_DATETIME'].astype(str).str.contains(target_time)
                
#                 filtered_df = df[mask]
                
#                 # Count filtered rows for reporting
#                 filtered_row_count = len(filtered_df)
#                 rows_filtered += filtered_row_count
                
#                 # Only write output if we have rows that match
#                 if filtered_row_count > 0:
#                     files_with_target += 1
                    
#                     # Construct output file path
#                     output_file_path = os.path.join(output_dir, file_name)
                    
#                     # Save the filtered DataFrame to a new parquet file
#                     filtered_df.to_parquet(output_file_path, index=False)
#                     print(f"  - Saved {filtered_row_count} rows with {target_time} to {output_file_path}")
#                 else:
#                     print(f"  - No rows with {target_time} found in this file")
#             else:
#                 print(f"  - Warning: INTERVAL_DATETIME column not found in {file_name}")
                
#         except Exception as e:
#             print(f"  - Error processing {file_name}: {str(e)}")
    
#     # Print summary statistics
#     print("\nProcessing complete!")
#     print(f"Processed {total_files} files with {total_rows_processed} total rows")
#     print(f"Found {rows_filtered} rows containing interval time {target_time}")
#     print(f"Found target time in {files_with_target} out of {total_files} files")
#     print(f"Filtered data saved to {output_dir}")

# if __name__ == "__main__":
#     # Directories
#     input_directory = "/Volumes/T7/bid-volume-filtered-2"
#     output_directory = "/Volumes/T7/bid-volume-filtered-3"
    
#     # Run the filter function for the 18:05:00 time period
#     filter_interval_time(
#         input_dir=input_directory,
#         output_dir=output_directory,
#         target_time='18:05:00'
#     )

Starting to filter data for interval time: 18:05:00
Output directory created/verified: /Volumes/T7/bid-volume-filtered-3
Found 1448 parquet files to process
Processing file 1/1448: PUBLIC_DVD_BIDPEROFFER_D_20090701.parquet
  - Saved 5828 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20090701.parquet
Processing file 2/1448: PUBLIC_DVD_BIDPEROFFER_D_20090801.parquet
  - Saved 5828 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20090801.parquet
Processing file 3/1448: PUBLIC_DVD_BIDPEROFFER_D_20090901.parquet
  - Saved 5622 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20090901.parquet
Processing file 4/1448: PUBLIC_DVD_BIDPEROFFER_D_20091001.parquet
  - Saved 5766 rows with 18:05:00 to /Volumes/T7/bid-volume-filtered-3/PUBLIC_DVD_BIDPEROFFER_D_20091001.parquet
Processing file 5/1448: PUBLIC_DVD_BIDPEROFFER_D_20091101.parquet
  - Saved 5580 rows with 18:05:00 to /Volumes/T7/bid-volume-f

In [ ]:
# Required to join DUIDs to firm names 
generator_info_df = static_table(table_name='Generators and Scheduled Loads', 
                              raw_data_location=raw_data_cache,
                              update_static_file=False)
generator_info_df

In [ ]:
# Gather all Parquet files in in a single dask data frame
all_files = glob.glob('/Volumes/T7/bid-volume-filtered-3/*.parquet')

# 2. Filter out any hidden dot-underscore files (._filename.parquet)
valid_files = [f for f in all_files if not os.path.basename(f).startswith('._')]

# 3. Print how many valid Parquet files were found
n_files = len(valid_files)
print(f"Found {n_files} valid Parquet file(s) in /Volumes/T7/bid-volume-filtered.\n")

# 4. Iterate over each valid file to show progress
for idx, file_path in enumerate(valid_files, start=1):
    file_name = os.path.basename(file_path)
    print(f"Processing file {idx} of {n_files}: {file_name}")

# 5. Read all valid files into a single Dask DataFrame
print("\nReading all valid Parquet files into a Dask DataFrame...")
ddf = dd.read_parquet(valid_files)
print("Done reading files into Dask DataFrame.\n")

# 7. Display columns and a small sample
print("Columns in the Dask DataFrame:", ddf.columns)
print("\nSample data from the Dask DataFrame:")
print(ddf.head())

NameError: name 'glob' is not defined

In [ ]:
ddf.head()

,SETTLEMENTDATE,DUID,BIDTYPE,OFFERDATE,MAXAVAIL,ENABLEMENTMIN,ENABLEMENTMAX,LOWBREAKPOINT,HIGHBREAKPOINT,BANDAVAIL1,BANDAVAIL2,BANDAVAIL3,BANDAVAIL4,BANDAVAIL5,BANDAVAIL6,BANDAVAIL7,BANDAVAIL8,BANDAVAIL9,BANDAVAIL10,INTERVAL_DATETIME
0,2021/04/01 00:00:00,BALBG1,LOWERREG,2021/03/26 13:30:57,30,0,30,30,30,0,0,0,0,0,0,0,0,0,30,2021/04/01 04:05:00
1,2021/04/01 00:00:00,BALBG1,RAISEREG,2021/03/26 13:30:57,30,0,30,0,0,0,0,0,0,10,10,10,0,0,0,2021/04/01 04:05:00
2,2021/04/01 00:00:00,BALBL1,LOWERREG,2021/03/26 13:38:40,30,0,30,0,0,0,0,0,0,0,0,0,0,0,30,2021/04/01 04:05:00
3,2021/04/01 00:00:00,BALBL1,RAISEREG,2021/03/26 13:38:40,28,0,30,30,30,0,30,0,0,0,0,0,0,0,0,2021/04/01 04:05:00
4,2021/04/01 00:00:00,BARKIPS1,LOWERREG,2021/03/17 01:00:43,0,9,210,126,210,0,0,0,0,0,0,0,0,0,108,2021/04/01 04:05:00


In [ ]:
# Filter the price bid data 
# Create the output directory if it doesn't already exist
output_dir = '/Volumes/T7/bid-price-filtered'
os.makedirs(output_dir, exist_ok=True)

# List all Feather files in the directory
file_list = glob.glob('/Volumes/T7/bid-price-data/*.feather')

for file in file_list:
    print(f"Processing {file}...")
    # Read the Feather file
    df = pd.read_feather(file)
    
    # Filter to keep rows where BIDTYPE is either "RAISEREG" or "LOWERREG"
    filtered_price_bids = df[(df["BIDTYPE"] == "RAISEREG") | (df["BIDTYPE"] == "LOWERREG")]

    # Print the first few rows of the filtered DataFrame
    print("Filtered DataFrame head:")
    print(filtered_price_bids.head())

    # Construct a new output file path by replacing the directory and file extension
    out_file = file.replace('bid-price-data', 'bid-price-filtered').replace('.feather', '.parquet')

    # Save the filtered DataFrame as a Parquet file
    filtered_price_bids.to_parquet(out_file, index=False)
    print(f"Finished processing {file}.\n")

Processing /Volumes/T7/bid-price-data/PUBLIC_DVD_BIDDAYOFFER_D_20210401.feather...
Filtered DataFrame head:
         SETTLEMENTDATE      DUID   BIDTYPE            OFFERDATE VERSIONNO  \
7   2021/04/01 00:00:00    BALBG1  RAISEREG  2021/03/26 13:30:57         1   
17  2021/04/01 00:00:00  BRAEMAR2  RAISEREG  2020/09/03 15:48:24         1   
18  2021/04/01 00:00:00  BRAEMAR3  LOWERREG  2020/09/03 15:49:58         1   
21  2021/04/01 00:00:00      BW01  RAISEREG  2021/03/31 03:55:20         1   
29  2021/04/01 00:00:00  DEVILS_G  LOWERREG  2021/04/01 01:30:23         1   

   PRICEBAND1 PRICEBAND2 PRICEBAND3 PRICEBAND4 PRICEBAND5 PRICEBAND6  \
7           0       7.89      13.35      28.89      61.89      92.89   
17          0          1          2          4          8         16   
18          0          1          2          4          8         16   
21          1        1.5         11       25.1         50        101   
29       0.01        2.5        3.8          9         14      

KeyboardInterrupt: 